# Train CBPR


In [1]:
# ! pip install "pandas<=2.3.2" "numpy" "torch<=2.5" "matplotlib" "seaborn" "matplotlib-venn" "datasets" "ipykernel" "recbole" "kmeans-pytorch" "sentence-transformers"

In [2]:
import numpy as np

# For NumPy 2.0 compatibility with RecBole 1.2
np.float_ = np.float64
np.int_ = np.int64
np.complex_ = np.complex128
np.unicode_ = np.str_

# Ensure logging on notebook works even on Colab
import logging
logging.getLogger().handlers.clear()

In [3]:
from typing import Any
import os
import torch
import torch.nn as nn
import pandas as pd
from recbole.config import Config
from recbole.data.dataloader import FullSortEvalDataLoader, AbstractDataLoader
from recbole.data import create_dataset, data_preparation
from recbole.model.abstract_recommender import GeneralRecommender
from recbole.model.init import xavier_normal_initialization
from recbole.model.loss import BPRLoss
from recbole.trainer import Trainer
from recbole.utils import init_seed, init_logger, InputType
from sentence_transformers import SentenceTransformer

/Users/nginyc/repos/amazon-item-recommender/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# --- Config ---
DATASET_NAME: str = "beauty"
DATA_DIR: str = "../data"
SEED = 67
DEVICE = "mps"  # Other options: "cpu", "cuda"

config_dict: dict[str, Any] = {
    "data_path": DATA_DIR,
    "dataset": DATASET_NAME,
    "USER_ID_FIELD": "user_id",
    "ITEM_ID_FIELD": "item_id",
    "benchmark_filename": ["train", "valid", "test"],
    "load_col": {
        "inter": ["user_id", "item_id"],
        "user": ["user_id", "cold"],
        "item": ["item_id", "cold", "title"],
    },
    "embedding_size": 64,
    "epochs": 100,
    "train_batch_size": 1024,
    "eval_batch_size": 409_600_000,
    "eval_args": {
        "split": None,
        "order": "TO",
        "mode": {"valid": "full", "test": "full"},
    },
    "metrics": ["NDCG", "Recall", "MRR"],
    "topk": [20],
    "valid_metric": "NDCG@20",
    "seed": SEED,
    "bge_embedding_path": f"{DATA_DIR}/{DATASET_NAME}/bge_title_embeddings.pt",
}

config: Config = Config(model="BPR", config_dict=config_dict)
config.final_config_dict["device"] = torch.device(DEVICE)

init_logger(config)
init_seed(SEED, reproducibility=True)

In [5]:
dataset = create_dataset(config)
train_data, valid_data, test_data = data_preparation(config, dataset)

/Users/nginyc/repos/amazon-item-recommender/.venv/lib/python3.12/site-packages/recbole/data/dataset/dataset.py:648: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  feat[field].fillna(value=0, inplace=True)
/Users/nginyc/repos/amazon-item-recommender/.venv/lib/python3.12/site-packages/recbole/data/dataset/dataset.py:650: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work becaus

## Precompute BGE title embeddings

In [6]:
# Suppress httpx INFO logs from SentenceTransformer
logging.getLogger('httpx').setLevel(logging.WARNING)

bge_path = config["bge_embedding_path"]
logger = logging.getLogger()

if not os.path.exists(bge_path):
    logger.info("Encoding %d item titles with BGE (BAAI/bge-base-en-v1.5)...", dataset.num("item_id"))

    title_tokens: torch.Tensor = dataset.item_feat["title"]
    id2token: dict[int, str] = {
        v: k for k, v in dataset.field2token_id["title"].items()
    }
    titles: list[str] = [id2token.get(tok.item(), "") for tok in title_tokens]

    bge_model = SentenceTransformer("BAAI/bge-base-en-v1.5", device=DEVICE)
    embs = bge_model.encode(titles, show_progress_bar=True, batch_size=256, normalize_embeddings=True)
    embs = torch.from_numpy(embs).float()

    torch.save(embs, bge_path)
    logger.info("BGE embeddings saved to %s", bge_path)
else:
    logger.info("BGE embeddings already exist at %s, loading cached.", bge_path)

22 Jun 15:40    INFO  BGE embeddings already exist at ../data/beauty/bge_title_embeddings.pt, loading cached.


## Define CBPR (VBPR-style)


In [7]:
class CBPR(GeneralRecommender):
    r"""CBPR: VBPR-style additive fusion with BGE text content.

    - User: separate collab and content embedding (2 x n_users x embedding_size)
    - Item: item_embedding[id] for collab + W @ bge[item] for content  (additive)
    - Score: dot(u_collab, item_embed[id]) + dot(u_content, W @ bge[item])  [no normalization]
    - Loss: BPR pairwise ranking loss
    """
    input_type = InputType.PAIRWISE

    def __init__(self, config, dataset):
        super().__init__(config, dataset)

        # Load precomputed BGE embeddings
        bge_path = config["bge_embedding_path"]
        embs = torch.load(bge_path, map_location=self.device)
        self.register_buffer("item_bge", embs)

        self.embedding_size = config["embedding_size"]
        self.content_embedding_size = 16

        # Separate user embeddings for collaborative and content pathways
        self.user_collab = nn.Embedding(self.n_users, self.embedding_size)
        self.user_content = nn.Embedding(self.n_users, self.content_embedding_size)

        # Per-item collaborative embedding (same as BPR)
        self.item_embedding = nn.Embedding(self.n_items, self.embedding_size)

        # Shared content projection (frozen BGE(768) -> learned space)
        self.item_proj = nn.Linear(768, self.content_embedding_size, bias=False)

        self.loss = BPRLoss()

        self.apply(xavier_normal_initialization)

        # Zero out padding item (index 0) so it never gets recommended
        with torch.no_grad():
            self.item_bge[0] = 0.0
            self.item_embedding.weight[0] = 0.0

    def get_user_collab(self, user):
        return self.user_collab(user)

    def get_user_content(self, user):
        return self.user_content(user)

    def calculate_loss(self, interaction):
        user = interaction[self.USER_ID]
        pos_item = interaction[self.ITEM_ID]
        neg_item = interaction[self.NEG_ITEM_ID]

        u_collab = self.get_user_collab(user)
        u_content = self.get_user_content(user)

        pos_collab = self.item_embedding(pos_item)
        neg_collab = self.item_embedding(neg_item)

        pos_content = self.item_proj(self.item_bge[pos_item])
        neg_content = self.item_proj(self.item_bge[neg_item])

        pos_score = torch.mul(u_collab, pos_collab).sum(-1) + torch.mul(u_content, pos_content).sum(-1)
        neg_score = torch.mul(u_collab, neg_collab).sum(-1) + torch.mul(u_content, neg_content).sum(-1)

        return self.loss(pos_score, neg_score)

    def predict(self, interaction):
        user = interaction[self.USER_ID]
        item = interaction[self.ITEM_ID]

        u_collab = self.get_user_collab(user)
        u_content = self.get_user_content(user)

        i_collab = self.item_embedding(item)
        i_content = self.item_proj(self.item_bge[item])

        return torch.mul(u_collab, i_collab).sum(-1) + torch.mul(u_content, i_content).sum(-1)

    @torch.no_grad()
    def full_sort_predict(self, interaction):
        user = interaction[self.USER_ID]
        u_collab = self.get_user_collab(user)
        u_content = self.get_user_content(user)

        score_collab = torch.matmul(u_collab, self.item_embedding.weight.t())
        score_content = torch.matmul(u_content, self.item_proj(self.item_bge).t())
        return (score_collab + score_content).view(-1)


## Train CBPR

In [8]:
model: CBPR = CBPR(config, train_data.dataset).to(config["device"])
trainer: Trainer = Trainer(config, model)

best_valid_score, best_valid_result = trainer.fit(train_data, valid_data)

/var/folders/vm/77wrgjgj5wzbyghx353b7gym0000gn/T/ipykernel_71187/2807630220.py:16: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  embs = torch.load(bge_path, map_location=sel

In [9]:
print(f"\nBest valid score: {best_valid_score:.4f}")
print("Best valid result:")
for metric, score in best_valid_result.items():
    print(f"  {metric}: {score:.4f}")


Best valid score: 0.0066
Best valid result:
  ndcg@20: 0.0066
  recall@20: 0.0122
  mrr@20: 0.0076


## Evaluate on test set

In [10]:
test_result: dict[str, float] = trainer.evaluate(test_data, load_best_model=False)

print("Test results (Overall):")
for metric, value in test_result.items():
    print(f"  {metric}: {value:.4f}")

Test results (Overall):
  ndcg@20: 0.0059
  recall@20: 0.0106
  mrr@20: 0.0071


## Evaluate by warm/cold user split

In [11]:
def evaluate_on_subset(
    data: AbstractDataLoader,
    mask: np.ndarray,
    label: str
):
    inter_feat = data.dataset.inter_feat
    cat_ds = data.dataset.copy(inter_feat[mask])
    cat_dl = FullSortEvalDataLoader(config, cat_ds, sampler=data._sampler)
    results = trainer.evaluate(cat_dl)
    rows.append({
        "Segment": label,
        "Interactions": int(mask.sum()),
        **results,
    })

rows = []

cold_to_label = {0.0: "warm", 1.0: "cold"}

def evaluate_by_column(entity_feat, id_field, inter_id_array, entity_name):
    id_to_cold = dict(zip(
        entity_feat[id_field].numpy(),
        entity_feat["cold"].numpy(),
    ))
    for cold_val, label in cold_to_label.items():
        ids = {eid for eid, c in id_to_cold.items() if c == cold_val}
        mask = np.isin(inter_id_array, list(ids))
        if not mask.any():
            print(f"  {entity_name}-{label}: no interactions — skipping")
            continue
        evaluate_on_subset(test_data, mask, f"{entity_name}-{label}")

# Evaluation by user segments
evaluate_by_column(
    dataset.user_feat, dataset.uid_field,
    test_data.dataset.inter_feat[dataset.uid_field].numpy(),
    "user"
)

# Evaluation by item segments
evaluate_by_column(
    dataset.item_feat, dataset.iid_field,
    test_data.dataset.inter_feat[dataset.iid_field].numpy(),
    "item"
)

# Cross-tabulation: user × item segments
uid_to_cold = dict(zip(
    dataset.user_feat[dataset.uid_field].numpy(),
    dataset.user_feat["cold"].numpy(),
))
iid_to_cold = dict(zip(
    dataset.item_feat[dataset.iid_field].numpy(),
    dataset.item_feat["cold"].numpy(),
))

uid_array = test_data.dataset.inter_feat[dataset.uid_field].numpy()
iid_array = test_data.dataset.inter_feat[dataset.iid_field].numpy()

for uc_val, uc_label in cold_to_label.items():
    for ic_val, ic_label in cold_to_label.items():
        uc_uids = {uid for uid, c in uid_to_cold.items() if c == uc_val}
        ic_iids = {iid for iid, c in iid_to_cold.items() if c == ic_val}
        mask = np.isin(uid_array, list(uc_uids)) & np.isin(iid_array, list(ic_iids))
        if not mask.any():
            print(f"  user-{uc_label}×item-{ic_label}: no interactions — skipping")
            continue
        evaluate_on_subset(test_data, mask, f"user-{uc_label}×item-{ic_label}")

# Display results sorted by NDCG
df_results = pd.DataFrame(rows).sort_values("ndcg@20", ascending=False)
display(df_results)


/Users/nginyc/repos/amazon-item-recommender/.venv/lib/python3.12/site-packages/recbole/trainer/trainer.py:583: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = tor

,Segment,Interactions,ndcg@20,recall@20,mrr@20
4,user-warm×item-warm,15736,0.0104,0.0187,0.0141
2,item-warm,88077,0.0093,0.0169,0.0101
6,user-cold×item-warm,72341,0.0091,0.0167,0.0096
0,user-warm,46742,0.0064,0.0104,0.0105
1,user-cold,150964,0.0063,0.0110,0.0073
3,item-cold,109629,0.0001,0.0002,0.0001
7,user-cold×item-cold,78623,0.0001,0.0002,0.0001
5,user-warm×item-cold,31006,0.0000,0.0000,0.0000
